In [0]:
from src.monitoring.feature_monitor import (
    calculate_feature_statistics
)

from src.monitoring.drift_detector import (
    detect_mean_drift
)

from src.monitoring.alert_generator import (
    generate_alerts
)

In [0]:
SOURCE_TABLE = (
    "tep_anomaly.served.training_base"
)

df = spark.table(
    SOURCE_TABLE
)

In [0]:
monitored_features = [
    "xmeas_1",
    "xmeas_2",
    "xmeas_3",
    "xmeas_1_delta1",
    "xmeas_2_delta1",
    "xmeas_3_delta1"
]


In [0]:
baseline_stats = (
    calculate_feature_statistics(
        df,
        monitored_features
    )
)

In [0]:
current_stats = (
    calculate_feature_statistics(
        df,
        monitored_features
    )
)

In [0]:
current_stats["xmeas_1"]["mean"] *= 1.5

In [0]:
drift_results = (
    detect_mean_drift(
        baseline_stats,
        current_stats,
        threshold=20
    )
)

drift_results

In [0]:
alerts = generate_alerts(
   drift_results
)

alerts

In [0]:
from pyspark.sql import Row

alert_rows = [
    Row(**alert)
    for alert in alerts
]

(
    spark.createDataFrame(alert_rows)
    .write
    .mode("append")
    .saveAsTable(
        "tep_anomaly.served.monitoring_alerts"
    )
)


In [0]:
display(
    spark.sql(
        """
        SELECT *
        FROM tep_anomaly.served.monitoring_alerts
        ORDER BY alert_timestamp DESC
        """
    )
)